# LungNet22 Model - Versi Baru (6 Classes)
Skenario A: Imbalanced Data (Tanpa GAN) - Disesuaikan dengan Panduan Project ILMED

Dataset: `chest-xray-preprocessed` (Kaggle Chest X-Ray 6 Classes yang telah di-preprocess offline dengan CLAHE)
- 6 Kelas: Covid-19, Emphysema, Normal, Pneumonia-Bacterial, Pneumonia-Viral, Tuberculosis
- Preprocessing Offline: Image Enhancement (CLAHE clipLimit=3, tileGridSize=(10,10)) telah diterapkan offline untuk menghindari bottleneck CPU saat training.
- Dataset: Penggabungan data Val & Test (sesuai instruksi Asdos)
- Backbone: VGG16 (Pretrained ImageNet)
- Penambahan: Block 6 & Block 7 (Dense 6)
- Evaluasi Lengkap: Akurasi/Loss, Confusion Matrix, Metrik per kelas (termasuk Specificity), ROC/AUC, Grad-CAM & Score-CAM


### 1. Impor Library & Cek Environment


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import glob
import cv2
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

print("TensorFlow Version:", tf.__version__)
# Cek ketersediaan GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPU is available:")
    for gpu in gpus:
        print("  -", gpu)
    # Aktifkan Mixed Precision untuk akselerasi training GPU
    from tensorflow.keras import mixed_precision
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print("Mixed precision (float16) enabled.")
else:
    print("GPU is NOT available. Running on CPU.")


### 2. Memuat Dataset Paths (Preprocessed Folder) & Menggabungkan Data Val + Test


In [ ]:
base_dir = './chest-xray-preprocessed' # Menggunakan dataset yang sudah di-CLAHE secara offline
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Fungsi pembantu untuk membuat dataframe berisi path file dan label
def get_data_df(directory):
    filepaths = []
    labels = []
    classes = os.listdir(directory)
    for cl in classes:
        cl_path = os.path.join(directory, cl)
        if os.path.isdir(cl_path):
            files = glob.glob(os.path.join(cl_path, "*"))
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    filepaths.append(f)
                    labels.append(cl)
    return pd.DataFrame({'Filepath': filepaths, 'Label': labels})

# Muat dataframe
train_df = get_data_df(train_dir)
val_df = get_data_df(val_dir)
test_df = get_data_df(test_dir)

# Gabungkan Val & Test sesuai instruksi Asdos
val_test_df = pd.concat([val_df, test_df], axis=0).reset_index(drop=True)

print(f"Jumlah data Train                  : {len(train_df)} file")
print(f"Jumlah data Val & Test (Gabungan)  : {len(val_test_df)} file")


### 3. Visualisasi Distribusi Kelas & Contoh Citra (Checklist Output Wajib)


In [ ]:
# Hitung distribusi per kelas
train_dist = train_df['Label'].value_counts().rename('Train')
val_test_dist = val_test_df['Label'].value_counts().rename('Val_Test_Merged')
dist_df = pd.concat([train_dist, val_test_dist], axis=1).fillna(0).astype(int)
dist_df['Total'] = dist_df['Train'] + dist_df['Val_Test_Merged']

print("--- Tabel Distribusi Data ---")
display(dist_df)

# Plot grafik batang distribusi data
plt.figure(figsize=(10, 5))
dist_df[['Train', 'Val_Test_Merged']].plot(kind='bar', stacked=False, figsize=(12, 6))
plt.title("Distribusi Jumlah Data per Kelas")
plt.xlabel("Kelas Penyakit")
plt.ylabel("Jumlah Gambar")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Tampilkan contoh gambar per kelas (sudah ter-CLAHE offline)
classes = train_df['Label'].unique()
fig, axes = plt.subplots(1, len(classes), figsize=(18, 5))

for i, cl in enumerate(sorted(classes)):
    sample_path = train_df[train_df['Label'] == cl]['Filepath'].iloc[0]
    img = Image.open(sample_path)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(cl)
    axes[i].axis('off')

plt.suptitle("Visualisasi Contoh Citra Chest X-Ray per Kelas (CLAHE)", fontsize=16, y=1.05)
plt.tight_layout()
plt.show()


### 4. Data Generators (Tanpa Bottleneck CPU)


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32 # Dikembalikan ke 32 karena batch size 128 menyebabkan bottleneck baca disk pada Windows

# Preprocessing online kustom dihilangkan karena gambar sudah diproses CLAHE secara offline.
# Sekarang, data generator hanya melakukan augmentation standar dan preprocess_input bawaan VGG16.
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input
)

# Flow from dataframe
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_test_generator = test_datagen.flow_from_dataframe(
    dataframe=val_test_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


### 5. Arsitektur Model LungNet22 (Fine-Tuned VGG16)


In [ ]:
# Load VGG16 Pretrained (ImageNet)
vgg16_base = tf.keras.applications.VGG16(
    weights='imagenet', 
    include_top=False, 
    input_shape=(224, 224, 3)
)

# Freeze VGG16 layers
vgg16_base.trainable = False

# Tambahkan Block 6 & 7 sesuai deskripsi paper
x = vgg16_base.output

# Block 6: 3x Conv2D (512 filter, kernel 3x3, ReLU) + GlobalAveragePooling2D
x = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block6_conv1')(x)
x = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block6_conv2')(x)
x = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block6_conv3')(x)
x = tf.keras.layers.GlobalAveragePooling2D(name='block6_pool')(x)

# Block 7: Flatten + Dense(6) + Softmax (dtype='float32' untuk mixed precision)
x = tf.keras.layers.Flatten(name='flatten')(x)
x = tf.keras.layers.Dense(6, name='dense_output')(x)
predictions = tf.keras.layers.Activation('softmax', dtype='float32', name='predictions')(x)

# Build Model
model = tf.keras.Model(inputs=vgg16_base.input, outputs=predictions)

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.000001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


### 6. Melatih Model (Training)


In [ ]:
EPOCHS = 100

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'lungnet22_skenario_a_best.h5',
    monitor='val_accuracy',
    save_best_only=True
)

history = model.fit(
    train_generator,
    validation_data=val_test_generator,
    epochs=EPOCHS,
    callbacks=[early_stopping, checkpoint]
)


### 7. Evaluasi Hasil Training: Kurva Akurasi & Loss


In [ ]:
# Plot Akurasi & Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Akurasi
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange', linewidth=2)
axes[0].set_title('Akurasi Model dari Epoch ke Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Akurasi')
axes[0].legend()
axes[0].grid(True)

# Plot Loss
axes[1].plot(history.history['loss'], label='Train Loss', color='blue', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', color='orange', linewidth=2)
axes[1].set_title('Loss Model dari Epoch ke Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


### 8. Prediksi & Confusion Matrix


In [ ]:
# Prediksi
val_test_generator.reset()
Y_pred = model.predict(val_test_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = val_test_generator.classes
class_names = list(val_test_generator.class_indices.keys())

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Skenario A (Imbalanced)')
plt.ylabel('Label Sebenarnya')
plt.xlabel('Label Prediksi')
plt.tight_layout()
plt.show()


### 9. Metrik Evaluasi Lengkap per Kelas (Presisi, Recall, Specificity, F1-Score)


In [ ]:
# Fungsi untuk menghitung Specificity per kelas
def calculate_specificity(cm):
    specificities = []
    num_classes = cm.shape[0]
    for i in range(num_classes):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - tp - fn - fp
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        specificities.append(specificity)
    return specificities

specs = calculate_specificity(cm)

# Hitung Precision, Recall, F1 dari classification_report
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

# Susun dalam DataFrame
metrics_data = []
for i, name in enumerate(class_names):
    metrics_data.append({
        'Kelas': name,
        'Accuracy': report_dict[name].get('accuracy', report_dict['accuracy']),
        'Precision': report_dict[name]['precision'],
        'Recall (Sensitivity)': report_dict[name]['recall'],
        'Specificity': specs[i],
        'F1-Score': report_dict[name]['f1-score']
    })

metrics_df = pd.DataFrame(metrics_data)
print("--- Tabel Metrik Evaluasi per Kelas ---")
display(metrics_df)

# Plot grafik batang perbandingan metrik
metrics_df.set_index('Kelas')[['Precision', 'Recall (Sensitivity)', 'Specificity', 'F1-Score']].plot(kind='bar', figsize=(12, 6))
plt.title("Perbandingan Metrik Evaluasi per Kelas — Skenario A")
plt.ylabel("Nilai Metrik (0.0 - 1.0)")
plt.xlabel("Kelas")
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### 10. Kurva ROC & Nilai AUC per Kelas


In [ ]:
# Binarize label true untuk perhitungan ROC multiclass (One-Vs-Rest)
y_true_bin = label_binarize(y_true, classes=np.arange(len(class_names)))

plt.figure(figsize=(10, 8))
for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], Y_pred[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'ROC {class_names[i]} (AUC = {roc_auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Tebakan Acak (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Receiver Operating Characteristic (ROC) Curve per Kelas')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


### 11. Visualisasi Interpretability: Grad-CAM & Score-CAM


In [ ]:
# Implementasi Grad-CAM
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Cast ke float32 untuk stabilitas numerik di mixed precision (float16)
    last_conv_layer_output = tf.cast(last_conv_layer_output[0], tf.float32)
    pooled_grads = tf.cast(pooled_grads, tf.float32)
    
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# Implementasi Score-CAM (Gradient-Free)
def make_scorecam_heatmap(img_array, model, last_conv_layer_name, pred_index=None, max_channels=64):
    feature_extractor = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output]
    )
    feature_maps = feature_extractor(img_array)[0] # Shape: (H_feat, W_feat, C_feat)
    
    # Cast ke float32 agar proses scoring dan normalisasi stabil dan tidak NaN
    feature_maps = tf.cast(feature_maps, tf.float32)
    
    if pred_index is None:
        preds = model(img_array)
        pred_index = tf.argmax(preds[0])
        
    h_feat, w_feat, c_feat = feature_maps.shape
    
    # Batasi jumlah channel yang dianalisis untuk efisiensi komputasi
    if c_feat > max_channels:
        stds = np.std(feature_maps.numpy(), axis=(0, 1))
        top_indices = np.argsort(stds)[::-1][:max_channels]
        selected_features = tf.gather(feature_maps, top_indices, axis=-1)
    else:
        selected_features = feature_maps
        top_indices = np.arange(c_feat)
        
    num_channels = selected_features.shape[-1]
    input_size = img_array.shape[1:3]
    
    # Resize feature maps ke ukuran input
    upsampled_features = tf.image.resize(selected_features, input_size, method='bilinear')
    
    # Normalisasi feature maps [0, 1]
    min_val = tf.reduce_min(upsampled_features, axis=(0, 1), keepdims=True)
    max_val = tf.reduce_max(upsampled_features, axis=(0, 1), keepdims=True)
    norm_features = (upsampled_features - min_val) / (max_val - min_val + 1e-10)
    
    # Buat masked images
    norm_features_expanded = tf.transpose(norm_features, perm=[2, 0, 1])
    norm_features_expanded = tf.expand_dims(norm_features_expanded, axis=-1)
    
    input_repeated = tf.repeat(img_array, num_channels, axis=0)
    masked_images = input_repeated * norm_features_expanded
    
    # Prediksi skor
    scores = []
    for start_idx in range(0, num_channels, 32):
        end_idx = min(start_idx + 32, num_channels)
        batch_masked = masked_images[start_idx:end_idx]
        batch_preds = model(batch_masked)
        batch_scores = batch_preds[:, pred_index]
        scores.extend(batch_scores.numpy())
        
    scores = np.array(scores)
    
    # Kombinasi linear feature maps berbobot
    weighted_features = selected_features.numpy() * scores[np.newaxis, np.newaxis, :]
    heatmap = np.sum(weighted_features, axis=-1)
    
    # ReLU & Normalisasi
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (np.max(heatmap) + 1e-10)
    
    return heatmap

# Fungsi untuk menumpangkan heatmap ke citra asli
def overlay_heatmap(img_path, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    # Cast ke float32 untuk menghindari error OpenCV resize biner float16
    heatmap = np.float32(heatmap)
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, colormap)
    superimposed_img = cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0)
    return cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB)


### 12. Menampilkan Hasil Grad-CAM & Score-CAM Berdampingan


In [ ]:
# Ambil 3 sampel acak dari data evaluasi
sample_rows = val_test_df.sample(n=3, random_state=42).reset_index(drop=True)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
last_conv_layer_name = 'block6_conv3'

for i, row in sample_rows.iterrows():
    img_path = row['Filepath']
    true_label = row['Label']
    
    # Load & preprocess
    img_pil = Image.open(img_path).resize((224, 224))
    img_arr = np.array(img_pil)
    if len(img_arr.shape) == 2:
        img_arr = np.stack([img_arr, img_arr, img_arr], axis=-1)
        
    # Di preprocessed folder, CLAHE sudah dilakukan secara offline.
    # Kita hanya perlu rescale dan centering untuk VGG16
    img_preprocessed = tf.keras.applications.vgg16.preprocess_input(img_arr.astype(np.float32))
    img_input = np.expand_dims(img_preprocessed, axis=0)
    
    # Predict
    preds = model.predict(img_input)
    pred_idx = np.argmax(preds[0])
    pred_label = class_names[pred_idx]
    
    # Heatmaps
    gradcam_heat = make_gradcam_heatmap(img_input, model, last_conv_layer_name, pred_index=pred_idx)
    scorecam_heat = make_scorecam_heatmap(img_input, model, last_conv_layer_name, pred_index=pred_idx, max_channels=64)
    
    # Overlays
    gradcam_img = overlay_heatmap(img_path, gradcam_heat)
    scorecam_img = overlay_heatmap(img_path, scorecam_heat)
    
    img_orig = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img_orig = cv2.resize(img_orig, (224, 224))
    
    # Plot original
    axes[i, 0].imshow(img_orig)
    axes[i, 0].set_title(f"Original Image\nTrue: {true_label}")
    axes[i, 0].axis('off')
    
    # Plot Grad-CAM
    axes[i, 1].imshow(gradcam_img)
    axes[i, 1].set_title(f"Grad-CAM\nPred: {pred_label} ({preds[0][pred_idx]*100:.1f}%)")
    axes[i, 1].axis('off')
    
    # Plot Score-CAM
    axes[i, 2].imshow(scorecam_img)
    axes[i, 2].set_title(f"Score-CAM\nPred: {pred_label} ({preds[0][pred_idx]*100:.1f}%)")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()
